# Working with data: numpy, plotting, pandas, a first model

Four libraries that most Python data work is built on:

- **numpy**: arrays of numbers, and fast calculations on all of them at once
- **matplotlib**: plots
- **pandas**: tables, with named columns, like a spreadsheet you program
- **scikit-learn**: machine learning, Wednesday's ideas as a library

None of them comes with Python. They're new in this repository: after pulling, run `uv sync` once
(it reads `pyproject.toml` and installs them).

The data files are next to this notebook: `cave.txt` (the cave from exercise 5) and
`titanic.csv` (Wednesday's passengers).

# Part 1: numpy

By convention, numpy is imported as `np`.

In [ ]:
import numpy as np

## Arrays are not lists

**Predict:**

In [ ]:
numbers = [1, 2, 3]
print(numbers * 2)
print(numbers + numbers)

Lists repeat and join (week 1). An **array** does arithmetic on every element instead.
**Predict:**

In [ ]:
values = np.array([1, 2, 3])
print(values * 2)
print(values + values)
print(values + 10)

In [ ]:
print(values * values)
print(values > 1)

Every operation goes element by element, and comparisons give an array of `True`/`False`.

An array holds one type of value only, its **dtype**:

In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)
print(np.array([1, 2.5, "3"]))

One float makes all of them floats; one string makes all of them strings. A list can mix types;
an array can't, and that's part of why it's fast: no checking the type of each element.

## Why bother: speed

**Predict:** both lines compute the sum of the squares of the first ten million numbers. How much
faster is numpy?

In [ ]:
import time

start = time.perf_counter()
total = 0
for n in range(10_000_000):
    total += n * n
print(total, round(time.perf_counter() - start, 3), "seconds with a loop")

start = time.perf_counter()
n = np.arange(10_000_000)
print((n * n).sum(), round(time.perf_counter() - start, 3), "seconds with numpy")

`np.arange` is `range` as an array. The loop in numpy runs in C, not in Python. The rule: **don't
loop over an array** in Python; say what to do with the whole array instead.

But look at the two totals: they're different, and numpy's is wrong. A Python `int` grows as large
as it needs to; a numpy `int64` has 64 bits, and when a result doesn't fit, it wraps around
**without an error**. The sum here is about 3 × 10²⁰, and `int64` ends at about 9 × 10¹⁸.


## Two dimensions: the cave

Exercise 5's cave, parsed as in `parse_grid` (a list of lists), then turned into an array.

In [ ]:
from pathlib import Path

rows = [[int(digit) for digit in line] for line in Path("cave.txt").read_text().split()]
cave = np.array(rows)
print(cave.shape)
print(cave)

`shape` is (rows, columns). numpy prints the corners and `...` for the rest.

**Predict:**

In [ ]:
print(cave[0, 0], cave[0][0])
print(cave[99, 99])

`cave[row, col]` is numpy's way to index; `cave[row][col]` also works, like a list of lists.
Slices work in every dimension. **Predict:** what are the shapes?

In [ ]:
print(cave[0].shape)
print(cave[:, 0].shape)
print(cave[:3, :5])

`cave[:, 0]` is column 0: all rows, column 0. With a list of lists that's a loop
(`[row[0] for row in rows]`).

Whole-array calculations, as with one dimension:

In [ ]:
print(cave.min(), cave.max(), cave.mean())
print(cave.sum(axis=0)[:5])
print(cave.sum(axis=1)[:5])

`axis=0` adds **down** the rows, giving one total per column; `axis=1` adds **along** each row.
(It's the axis that disappears.)

## Masks: arrays of True and False

**Predict:** what's printed?

In [ ]:
nines = cave == 9
print(nines.shape)
print(nines[:2, :6])
print(nines.sum())

`True` counts as 1 and `False` as 0 (like `sum([True, True, False])`), so `.sum()` of a mask counts.
A mask can also **select**:

In [ ]:
low = cave[cave <= 2]
print(low)
print(low.shape)

Selecting with a mask gives a flat array of the matching values. Two conditions: **Predict:**

In [ ]:
print(((cave >= 6) & (cave <= 7)).sum())

In [ ]:
print((cave >= 6 and cave <= 7).sum())

`and` needs one `True` or `False`, and an array of 10 000 of them isn't one. For arrays, write `&`
(and), `|` (or) and `~` (not), with brackets around each comparison: `&` binds tighter than `<=`.

## Slices are views

**Predict:** week 3, lists. What's `a` at the end?

In [ ]:
a = [1, 2, 3, 4]
b = a[1:3]
b[0] = 99
print(a)

A list slice is a copy. **Predict:** the same with an array.

In [ ]:
a = np.array([1, 2, 3, 4])
b = a[1:3]
b[0] = 99
print(a)

An array slice is a **view**: a second label on part of the same memory. numpy does this because
copying large arrays is slow. If you want a copy, say so: `b = a[1:3].copy()`. (A mask selection,
`a[a > 1]`, is always a copy.)

## Making arrays

In [ ]:
print(np.zeros(3))
print(np.ones((2, 3), dtype=int))
print(np.linspace(0, 1, 5))
print(np.arange(6).reshape(2, 3))

`linspace(start, stop, count)` includes the end; `arange` is like `range`. `reshape` gives the
same numbers a new shape.

## Random numbers

In [ ]:
rng = np.random.default_rng(seed=1)
dice = rng.integers(1, 7, size=10)
print(dice)

**Predict:** roll 600 000 times. How often does each number come up, roughly?

In [ ]:
rolls = rng.integers(1, 7, size=600_000)
for face in range(1, 7):
    print(face, (rolls == face).sum())

About 100 000 each, never exactly. `integers(1, 7)` stops before 7, like `range`. With a seed, the
"random" numbers are the same on every run: as with Wednesday's `random.Random(1)`.

# Part 2: plotting with matplotlib

The usual import:

In [ ]:
import matplotlib.pyplot as plt

## A line plot

From two weeks ago: how many calls does the recursive `ways(n)` make? With numpy, `n²` for a whole
range is one line.

In [ ]:
def calls_of_ways(n):
    if n <= 1:
        return 1
    return 1 + calls_of_ways(n - 1) + calls_of_ways(n - 2)


ns = np.arange(1, 21)
calls = np.array([calls_of_ways(n) for n in ns])
print(calls[:10])

In [ ]:
fig, ax = plt.subplots()
ax.plot(ns, calls)
ax.set_xlabel("n")
ax.set_ylabel("calls")
ax.set_title("Calls of ways(n)")
plt.show()

`plt.subplots()` makes a **figure** (the whole picture) and one **axes** (one plot in it).
Everything about that plot is a method of `ax`. `plt.show()` shows the figure: below the cell in a
notebook, in a window when you run a script.

**Predict:** we add `n` and `n²` to the same plot. Which line is on top at `n = 5`? At `n = 20`?

In [ ]:
fig, ax = plt.subplots()
ax.plot(ns, calls, label="calls of ways(n)")
ax.plot(ns, ns**2, label="n²")
ax.plot(ns, ns, label="n")
ax.set_xlabel("n")
ax.legend()
plt.show()

**Predict:** what do the three lines look like on a logarithmic y axis, where each step up
multiplies by 10?

In [ ]:
ax.set_yscale("log")
fig

On a log scale, "×1.6 for every step" is a **straight line**, and `n` and `n²` bend over. A straight
line on a log scale is how you spot this kind of growth in measurements. (`fig` on its own in the
last line shows the figure again.)

## Points, bars, histograms

In [ ]:
xs = rng.normal(0, 1, size=300)
ys = xs + rng.normal(0, 0.5, size=300)

fig, ax = plt.subplots()
ax.scatter(xs, ys, s=10)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

`rng.normal(0, 1, size=300)`: 300 random numbers around 0, most between -2 and 2. `ys` is `xs`
plus some noise, so the points lie around a line. `s` is the size of the points.

**Predict:** which bar is the tallest?

In [ ]:
faces, counts = np.unique(rolls, return_counts=True)

fig, ax = plt.subplots()
ax.bar(faces, counts)
ax.set_xlabel("face")
plt.show()

All about the same. **Predict:** the same bars, but the y axis only shows 99 500 to 101 000.

In [ ]:
ax.set_ylim(99_500, 101_000)
fig

Now 2 looks like the winner by far. Same numbers: a difference of 1%. Check where the axis
starts, in other people's plots and in your own.

A **histogram** counts how many values fall into each range ("bin"):

In [ ]:
fig, ax = plt.subplots()
ax.hist(xs, bins=20)
plt.show()

## Several plots in one figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ns, calls)
axes[0].set_title("calls")
axes[1].hist(xs, bins=20)
axes[1].set_title("normal")
fig.tight_layout()
plt.show()

`plt.subplots(1, 2)` makes one row of two plots; `axes` holds both (it's a numpy array). `figsize`
is in inches. `tight_layout` moves the plots so the labels don't overlap.

In [ ]:
fig.savefig("two_plots.png")

**Predict:** where is the file now?

`savefig` writes to the working directory (week 4), so where the file ends up depends on where you
started Python. In a script, build the path from `Path(__file__).parent`.

## A grid as a picture

**Predict:** `imshow` draws each number as a coloured square. In which corner of the picture is
`cave[0, 99]`?

In [ ]:
fig, ax = plt.subplots()
image = ax.imshow(cave)
fig.colorbar(image, ax=ax)
plt.show()

Row 0 is at the **top**, as when you print the grid, not at the bottom as in a usual plot. The
colour bar says which colour stands for which number. Now you can see the tunnels of low risk that
the cave was built with.

**Predict:** we draw a line from `(0, 0)` to `(99, 20)` on top. Where does it go?

In [ ]:
ax.plot([0, 99], [0, 20], color="red")
fig

`ax.plot(xs, ys)` takes **x first**: x is the column, y the row. So `(99, 20)` is column 99, row 20,
while the array is indexed `cave[row, col]`, the other way round. Mixing up `(x, y)` and
`[row][col]` is the classic bug with grids: when a picture comes out mirrored along the diagonal,
look there first.

## The short form

Many examples online skip `fig` and `ax`:

In [ ]:
plt.plot(ns, calls)
plt.title("Calls of ways(n)")
plt.show()

`plt.plot` draws on the "current" plot, whichever that is. That's fine for one quick plot; with
several plots in one figure, `fig, ax = plt.subplots()` makes it clear which plot each line goes
to. We use that form.

# Part 3: pandas

A numpy array is a grid of numbers of one type. A **DataFrame** is a table: every column has a
name and its own type, and rows have labels too. By convention, pandas is imported as `pd`.

In [ ]:
import pandas as pd

passengers = pd.read_csv("titanic.csv")
passengers

One line instead of Wednesday's `csv.DictReader`, and the quotes around names with commas are
handled. Empty fields become `NaN` ("not a number"), pandas' marker for a missing value.

In [ ]:
print(passengers.shape)
print(passengers.columns)

In [ ]:
passengers.dtypes

Unlike Wednesday, `Survived` and `Pclass` are numbers now: `read_csv` looks at each column and
picks a type. `Age` is a float, because `NaN` is a float. Text columns are `str`.

## Columns

In [ ]:
ages = passengers["Age"]
print(type(ages))
ages.head()

A column is a **Series**: a numpy array with a name and row labels (the **index**, 0 to 890 here).
Arithmetic and comparisons work element by element, as with arrays. **Predict:**

In [ ]:
print(ages.max(), ages.min())
print((ages < 1).sum())

**Predict:** `Survived` is 0 or 1. What does its mean tell you?

In [ ]:
print(passengers["Survived"].mean())

The mean of a 0/1 column is the fraction of 1s: 38% survived. Wednesday's baseline (62%) is
1 minus this.

In [ ]:
passengers["Sex"].value_counts()

In [ ]:
passengers.describe()

`describe` summarizes every number column. Look at `count`: `Age` has only 714 of 891.

## Missing values

**Predict:** passengers under 16 plus passengers 16 or older: is that everyone?

In [ ]:
children = (passengers["Age"] < 16).sum()
others = (passengers["Age"] >= 16).sum()
print(children, others, children + others)

Any comparison with `NaN` is `False`, so the 177 passengers without an age are in neither group.

In [ ]:
print(passengers["Age"].isna().sum())
print(float("nan") == float("nan"))

`NaN` isn't even equal to itself. Test for it with `isna()`, never with `==`. `mean`, `max` and
friends skip `NaN` (so the mean age is the mean of the 714 known ones).

## Selecting rows

A mask selects rows, as with arrays:

In [ ]:
women = passengers[passengers["Sex"] == "female"]
print(len(women))
women.head()

**Predict:** the survival rate of women, then of women in third class.

In [ ]:
print(women["Survived"].mean())
third_class_women = passengers[(passengers["Sex"] == "female") & (passengers["Pclass"] == 3)]
print(third_class_women["Survived"].mean())

Wednesday's 72 of 144. As with arrays: `&`, and brackets around each comparison.

The row labels stay with the rows:

In [ ]:
third_class_women.head(3)

`loc` picks rows **and** columns by label; `iloc` by position, like a list:

In [ ]:
print(passengers.loc[1, "Name"])
print(passengers.iloc[0, 3])
passengers.loc[passengers["Age"] < 1, ["Name", "Age", "Survived"]]

## Sorting

In [ ]:
passengers.sort_values("Fare", ascending=False).head()

**Predict:** after sorting, what's `passengers.iloc[0]["Name"]`?

In [ ]:
print(passengers.iloc[0]["Name"])

Still Mr. Braund: `sort_values` returns a **new** DataFrame and leaves `passengers` alone, like
`sorted(...)`, not like `list.sort()`. Most pandas methods work that way: assign the result if you
want to keep it.

## Changing a selection

**Predict:** we set every age in `women` to 0. What happens to `passengers`?

In [ ]:
women = passengers[passengers["Sex"] == "female"]
women["Age"] = 0
print(women["Age"].max())
print(passengers["Age"].max())

`passengers` is unchanged. Since pandas 3, a selection behaves like a copy: changing it never
changes the table it came from (unlike a numpy slice). Older pandas versions sometimes did change
the original, and printed a `SettingWithCopyWarning`, which you'll still see in many answers online.

To change the table itself, use `loc` on it directly:

In [ ]:
fixed = passengers.copy()
fixed.loc[fixed["Age"] < 1, "Age"] = 1
print((fixed["Age"] < 1).sum(), (passengers["Age"] < 1).sum())

## Grouping

Wednesday's `learn` counted survivors per group with a dict. pandas does it in one line:

In [ ]:
passengers.groupby("Sex")["Survived"].mean()

In [ ]:
passengers.groupby(["Sex", "Pclass"])["Survived"].agg(["mean", "sum", "count"])

"Group by sex and class, take the `Survived` column of each group, compute its mean, sum and count":
the 72 of 144 again.

## New columns, and `x if condition else y`

A new column from a calculation:

In [ ]:
passengers["Relatives"] = passengers["SibSp"] + passengers["Parch"]
passengers[["Name", "SibSp", "Parch", "Relatives"]].head()

`SibSp` is siblings and spouses on board, `Parch` parents and children.

For Wednesday's `age_group`, we need a choice per passenger. Python has a short form of `if`/`else`
that gives a **value**, the **conditional expression**. **Predict:**

In [ ]:
age = 12
group = "child" if age < 16 else "adult"
print(group)
print("even" if 7 % 2 == 0 else "odd")

`x if condition else y` is `x` when the condition is true, otherwise `y`. It's the same as:

```python
if age < 16:
    group = "child"
else:
    group = "adult"
```

Use it for short choices between two values; for anything longer, the `if` statement is easier to
read.

`apply` calls a function on every value of a column and collects the results (like
`[f(x) for x in column]`). With a `lambda` (week 5):

In [ ]:
passengers["Group"] = passengers["Age"].apply(lambda age: "child" if age < 16 else "adult")
passengers["Group"].value_counts()

**Predict:** that's 83 children and 808 adults. Is that right?

In [ ]:
passengers.loc[passengers["Age"].isna(), ["Name", "Age", "Group"]].head()

`NaN < 16` is `False`, so everyone without an age became an "adult". A missing value got a real
answer without any error: the most common data bug there is. A version that says so:

In [ ]:
def age_group(age):
    if pd.isna(age):
        return "unknown"
    return "child" if age < 16 else "adult"


passengers["Group"] = passengers["Age"].apply(age_group)
passengers["Group"].value_counts()

In [ ]:
passengers.groupby(["Sex", "Group"])["Survived"].mean()

Wednesday's "sex, child" rule, as a table: boys 0.525, so the rule said `"1"` for them.

## Text columns

Text methods for a whole column are under `.str`:

In [ ]:
print(passengers["Name"].str.len().max())
passengers[passengers["Name"].str.contains("Rev.", regex=False)][["Name", "Survived"]]

`.str.contains`, `.str.split`, `.str.upper`, `.str.strip`: the string methods from week 4, for every
row at once.

## Plotting from pandas

A Series or DataFrame can plot itself, onto an `ax` you made:

In [ ]:
fig, ax = plt.subplots()
passengers["Age"].plot.hist(bins=20, ax=ax)
ax.set_xlabel("age")
plt.show()

In [ ]:
rates = passengers.groupby("Pclass")["Survived"].mean()

fig, ax = plt.subplots()
rates.plot.bar(ax=ax)
ax.set_ylabel("survival rate")
plt.show()

**Predict:** we plot age against fare, one point per passenger. What do you expect to see?

In [ ]:
fig, ax = plt.subplots()
ax.scatter(passengers["Age"], passengers["Fare"], s=8, c=passengers["Survived"], cmap="coolwarm")
ax.set_xlabel("age")
ax.set_ylabel("fare")
plt.show()

Not much of a pattern with age; a few very high fares, mostly survivors (red). Passengers without
an age are simply missing from the plot, without a warning.

## Writing results

In [ ]:
summary = passengers.groupby(["Sex", "Pclass"])["Survived"].mean()
summary.to_csv("survival_by_sex_and_class.csv")
print(Path("survival_by_sex_and_class.csv").read_text())

# Part 4: a model with scikit-learn

Wednesday's `learn` put passengers into groups we chose. A **decision tree** chooses the groups
itself: it finds the one question that splits the passengers best ("female?"), then the best next
question within each half, and so on. `max_depth` says how many questions in a row it may ask: how
fine the groups get.

## Features as numbers

scikit-learn wants numbers. Sex becomes 0/1, and the missing ages need a decision: here, -1, a
value no real passenger has, so the tree can treat "unknown" as its own group.

In [ ]:
features = pd.DataFrame(
    {
        "female": (passengers["Sex"] == "female").astype(int),
        "pclass": passengers["Pclass"],
        "age": passengers["Age"].fillna(-1),
    }
)
label = passengers["Survived"]
features.head()

## Train and test

In [ ]:
from sklearn.model_selection import train_test_split

train_features, test_features, train_label, test_label = train_test_split(
    features, label, test_size=191, random_state=1
)
print(train_features.shape, test_features.shape)
print("baseline on test:", (test_label == 0).mean())

`train_test_split` shuffles and splits, as we did by hand; `random_state` is the seed. It returns
four things, in this order (check the documentation every time; swapping two of them is a classic
silent bug). It shuffles differently from Wednesday's `random.Random(1)`, so this test set has
different passengers: its baseline is 58%, not 63%.

## Fit, predict, score

A tree that may ask one question:

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=1, random_state=1)
tree.fit(train_features, train_label)
predictions = tree.predict(test_features)
print(predictions[:10])
print((predictions == test_label).mean())

`fit` is Wednesday's `learn`; `predict` is calling the model on each passenger. `score` does the
last line for you. **Predict:** which question did it pick?

In [ ]:
print(export_text(tree, feature_names=list(features.columns)))
print("train:", tree.score(train_features, train_label))
print("test: ", tree.score(test_features, test_label))

`female <= 0.50` means "male". `class: 0` is the **predicted label** (scikit-learn calls the
labels "classes"; nothing to do with `pclass`). One question, and it's Wednesday's `women_survive`.

**Predict:** two questions. Better on test?

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=1)
tree.fit(train_features, train_label)
print(export_text(tree, feature_names=list(features.columns)))
print("train:", tree.score(train_features, train_label))
print("test: ", tree.score(test_features, test_label))

Second question: the class. For men, both sides still say 0: the tree splits where the groups get
**purer**, even when the answer doesn't change. For women, third class now gets 0, and that's
worse on this test set (0.749 instead of 0.780): Wednesday's women in third class, about half of
whom survived. Six test passengers make that difference.

**Predict:** no limit on depth (`max_depth=None`). Train and test accuracy?

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=1)
deep_tree.fit(train_features, train_label)
print("depth:", deep_tree.get_depth(), "leaves:", deep_tree.get_n_leaves())
print("train:", deep_tree.score(train_features, train_label))
print("test: ", deep_tree.score(test_features, test_label))

122 groups, and 89% on the passengers it learned from: it has memorized a good part of them (two
passengers with the same sex, class and age but different fates can't both be right, so it can't
reach 100%). On test, it's no better than the one-question tree. The gap between train and test
is the sign of overfitting. `max_depth` is a **hyperparameter**: a choice made before training,
and with 191 test passengers, differences of a few percent between choices may be luck.

## A model is a function, still

In [ ]:
someone = pd.DataFrame({"female": [1, 0], "pclass": [3, 1], "age": [30, 8]})
print(tree.predict(someone))
print(deep_tree.predict(someone))

A woman of 30 in third class, and a boy of 8 in first class. The deep tree says the boy survives:
it asks about age; the two-question tree never does.

## Summary

- **numpy:** arrays of one type; operations on all elements at once; masks (`a > 1`, `&`, `|`,
  `.sum()` to count); `shape`, `axis`; slices are **views**.
- **matplotlib:** `fig, ax = plt.subplots()`, then `ax.plot`, `ax.scatter`, `ax.bar`, `ax.hist`,
  `ax.imshow` (row 0 at the top; x is the column); labels, legend, log scale; `fig.savefig`.
- **pandas:** `read_csv`; a DataFrame of named columns, each a Series; masks select rows; `loc`
  and `iloc`; `NaN` and `isna()`; `groupby(...)[...].mean()`; methods return new tables;
  `apply` with a function; `.str`; `.plot`.
- **`x if condition else y`:** a value, not a statement.
- **scikit-learn:** features as numbers, `train_test_split`, `fit`, `predict`, `score`,
  `export_text`, `max_depth` against overfitting. Always compare with the baseline and with the
  simplest model.

In [ ]:
# tidy up the files this notebook wrote
Path("two_plots.png").unlink()
Path("survival_by_sex_and_class.csv").unlink()